# BACKTESTING_272 — Stratégie Momentum Dual Minimum Variance (Markowitz)
 
**Univers** : STOXX Europe 600  
**Période** : 2016-12-30 → 2026-02-20  
**Benchmark** : ESTRON Index


## 1. Importation des Libraries <a id='1'></a>

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

import pandas as pd
import numpy as np
from IPython.display import display


from backtesting import DataLoader, BacktestEngine, ReportingEngine


## 2. Paramètres du portefeuille <a id='1'></a>

In [2]:
INITIAL_CAPITAL = 1_000_000

TRANSACTION_COST = 0.0005

SELECTION_QUANTILE = 0.40

METHODS = ["min_variance"]

"""
- SECTOR_MIN : nombre minimum de poids par secteur
- SECTOR_MAX : nombre maximum de poids par secteur
"""
SECTOR_MIN = None
SECTOR_MAX = None

print(f"Capital initial   : {INITIAL_CAPITAL:,.0f} €")
print(f"Frais transaction : {TRANSACTION_COST * 10_000:.1f} bps")
print(f"Sélection         : top/bottom {SELECTION_QUANTILE*100:.0f}%")
print(f"Méthodes          : {METHODS}")
print(f"Contraintes       : SECTOR_MIN={SECTOR_MIN}, SECTOR_MAX={SECTOR_MAX}")

Capital initial   : 1,000,000 €
Frais transaction : 5.0 bps
Sélection         : top/bottom 40%
Méthodes          : ['min_variance']
Contraintes       : SECTOR_MIN=None, SECTOR_MAX=None



## 3. Chargement des données <a id='1'></a>

In [3]:
"""
- start_date : date de début du backtest (format "YYYY-MM-DD")
- end_date : date de fin du backtest (format "YYYY-MM-DD")
"""

loader = DataLoader(start_date="2016-12-30", end_date="2026-02-20")

"""
- initial_capital    : implémente le capital initinial du portefeuille
- transaction_cost   : implémente le coût de transaction par trade 
- selection_quantile : implémente le quantile utilisé pour sélectionner les positions long/short au travers des signaux
- sector_min         : implémente le poids minimum autorisé par secteur (contrainte de diversification)
- sector_max         : implémente le poids maximum autorisé par secteur (contrainte de diversification)
"""

engine = BacktestEngine(
    data_loader=loader,
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=TRANSACTION_COST,
    selection_quantile=SELECTION_QUANTILE,
    sector_min=SECTOR_MIN,
    sector_max=SECTOR_MAX,
)

results = {}

print(f"\nDonnées chargées : {len(loader.rebalancing_dates)} dates de rebalancement")
print(f"Tickers disponibles : {loader.prices.shape[1]}")
print(f"Période des prix : {loader.prices.index[0].date()} → {loader.prices.index[-1].date()}")

[DataLoader] Chargement des données en cours...
[DataLoader] Prêt : 111 dates de rebalancement (2016-12-30 → 2026-02-20).

Données chargées : 111 dates de rebalancement
Tickers disponibles : 980
Période des prix : 2015-09-01 → 2026-02-20



## 4. Backtesting <a id='1'></a>

In [4]:
"""
- Choix : min_variance
"""
METHOD = "min_variance"

if METHOD in METHODS:
    print(f"{'='*60}")
    print(f"  Lancement : {METHOD}")
    print(f"{'='*60}")
    results[METHOD] = engine.run(METHOD)
    nav_final = results[METHOD].nav.iloc[-1]
    total_ret = nav_final / INITIAL_CAPITAL - 1
    print(f"\n  NAV finale : {nav_final:,.0f} €  |  Rendement : {total_ret:.2%}")
else:
    print(f"'{METHOD}' non sélectionné dans METHODS — cellule ignorée.")

  Lancement : min_variance

[BacktestEngine] Démarrage — méthode : min_variance
  [  1/111] 2016-12-30 NAV=   1,005,856€  Long=232  Short=232
  [  2/111] 2017-01-31 NAV=   1,018,754€  Long=230  Short=230
  [  3/111] 2017-02-28 NAV=   1,015,860€  Long=231  Short=231
  [  4/111] 2017-03-31 NAV=   1,029,964€  Long=230  Short=230
  [  5/111] 2017-04-28 NAV=   1,026,345€  Long=230  Short=230
  [  6/111] 2017-05-31 NAV=   1,026,870€  Long=229  Short=229
  [  7/111] 2017-06-30 NAV=   1,020,285€  Long=231  Short=231
  [  8/111] 2017-07-31 NAV=   1,022,090€  Long=231  Short=231
  [  9/111] 2017-08-31 NAV=   1,010,909€  Long=231  Short=231
  [ 10/111] 2017-09-29 NAV=   1,019,689€  Long=230  Short=230
  [ 11/111] 2017-10-31 NAV=   1,034,569€  Long=203  Short=203
  [ 12/111] 2017-11-30 NAV=   1,028,105€  Long=232  Short=232
  [ 13/111] 2017-12-29 NAV=   1,041,222€  Long=231  Short=231
  [ 14/111] 2018-01-31 NAV=   1,055,910€  Long=232  Short=232
  [ 15/111] 2018-02-28 NAV=   1,046,321€  Long=167  


## 5. Reporting <a id='1'></a>

In [5]:
METHOD_TO_REPORT = METHODS.pop(0) 

"""
- AS_OF_DATE       : date d’arrêté des métriques de performance (format "YYYY-MM-DD")
"""
AS_OF_DATE = "2026-01-30"

reporter = ReportingEngine(results[METHOD_TO_REPORT], loader)
as_of = pd.Timestamp(AS_OF_DATE)

In [6]:

metrics = reporter.compute_metrics(as_of)


def format_metric(v, key=None):
    """
    - si la valeur est "--", elle est retournée telle qu'une valeur manquante,
    - si la valeur est un float :
         si la métrique est un ratio (Sharpe, IR, Corrélation, etc.) -> format décimal
         sinon (rendements, vol, drawdown, TC%) -> format en %
    - sinon, la valeur est convertie en chaîne de caractères.
    """
    if v == "--":
        return "--"

    if isinstance(v, float):

        if key is not None and ("(€)" in key or "€" in key):
            return f"{v:,.2f}"


        ratio_keys = (
            "Sharpe", "Sortino", "Information Ratio",
            "Calmar", "Corrélation", "Tracking Error"
        )
        if key is not None and any(rk in key for rk in ratio_keys):
            return f"{v:.4f}"


        return f"{v*100:.2f}%"

    return str(v)


"""
- Serie pandas indéxée par le nom des indicateurs et datée à la date de reporting
"""

metrics_series = pd.Series({
    k: format_metric(v, k) for k, v in metrics.items()
}, name=str(as_of.date()))
metrics_series.index.name = "Métrique"

categories = {
    "Performance": [
        "Rendement cumulé (période)", "Rendement cumulé 1 an",
        "Rendement cumulé 2 ans", "Rendement cumulé YTD", "Rendement cumulé MTD",
        "Rendement annualisé (période)",
        "CAGR (période)", "CAGR 1 an", "CAGR 2 ans", "CAGR YTD", "CAGR MTD",
    ]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Performance
──────────────────────────────────────────────────
  Rendement cumulé (période)                          32.13%
  Rendement cumulé 1 an                                7.37%
  Rendement cumulé 2 ans                              12.11%
  Rendement cumulé YTD                                 1.53%
  Rendement cumulé MTD                                 1.53%
  Rendement annualisé (période)                        3.06%
  CAGR (période)                                       2.12%
  CAGR 1 an                                            8.16%
  CAGR 2 ans                                           7.12%
  CAGR YTD                                            20.28%
  CAGR MTD                                            20.28%


In [7]:
categories = {
    "Risque & Ratios": [
        "Sharpe ratio (période)", "Sharpe ratio 1 an",
        "Sortino ratio (période)", "Sortino ratio 1 an",
        "Tracking Error (période)", "Tracking Error 1 an", "Tracking Error 3 ans",
        "Information Ratio (période)", "Information Ratio 1 an",
        "Information Ratio 2 ans",
        "Calmar ratio (période)",
        "VaR 95% (période)", "VaR 95% 1 an",
        "CVaR 95% (période)", "CVaR 95% 1 an",
        "Volatilité (période)", "Volatilité 1 an", "Volatilité 2 ans",
        "Max Drawdown (période)", "Max Drawdown 1 an",
        "Max Drawdown 2 ans", "Max Drawdown YTD", "Max Drawdown MTD",
    ]}


for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>20}")


──────────────────────────────────────────────────
  Risque & Ratios
──────────────────────────────────────────────────
  Sharpe ratio (période)                                      0.3228
  Sharpe ratio 1 an                                           0.7660
  Sortino ratio (période)                                     0.4264
  Sortino ratio 1 an                                          1.0709
  Tracking Error (période)                                    0.0818
  Tracking Error 1 an                                         0.0790
  Tracking Error 3 ans                                        0.0663
  Information Ratio (période)                                 0.3228
  Information Ratio 1 an                                      0.7660
  Information Ratio 2 ans                                     0.6049
  Calmar ratio (période)                                      0.2050
  VaR 95% (période)                                            0.80%
  VaR 95% 1 an                                     

In [8]:

categories = {
       "Benchmark (ESTER)": [
        "Corrélation avec benchmark (période)",
        "Benchmark - Rendement cumulé (période)",
        "Benchmark - Rendement cumulé 1 an",
        "Benchmark - Rendement cumulé 2 ans",
        "Benchmark - Rendement cumulé YTD",
        "Benchmark - Rendement cumulé MTD",
        "Benchmark - Rendement annualisé (période)",
    ],
    "Coûts de transaction": ["TC total (€)", "TC total (%)"]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Benchmark (ESTER)
──────────────────────────────────────────────────
  Corrélation avec benchmark (période)                0.0028
  Benchmark - Rendement cumulé (période)               8.38%
  Benchmark - Rendement cumulé 1 an                    2.15%
  Benchmark - Rendement cumulé 2 ans                   5.92%
  Benchmark - Rendement cumulé YTD                     0.16%
  Benchmark - Rendement cumulé MTD                     0.16%
  Benchmark - Rendement annualisé (période)            1.25%

──────────────────────────────────────────────────
  Coûts de transaction
──────────────────────────────────────────────────
  TC total (€)                                     89,773.70
  TC total (%)                                         8.98%


In [9]:
all_metrics_df = reporter.compute_all_metrics()
display(all_metrics_df.tail(5))

,Rendement cumulé (période),Rendement cumulé 1 an,Rendement cumulé 2 ans,Rendement cumulé YTD,Rendement cumulé MTD,Rendement annualisé (période),CAGR (période),CAGR 1 an,CAGR 2 ans,CAGR YTD,...,Corrélation avec benchmark (période),Benchmark - Rendement cumulé (période),Benchmark - Rendement cumulé 1 an,Benchmark - Rendement cumulé 2 ans,Benchmark - Rendement cumulé YTD,Benchmark - Rendement cumulé MTD,Benchmark - Rendement annualisé (période),TC total (€),TC total (%),PnL (€)
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-31,0.278038,0.05281,0.085009,0.050092,0.021588,0.027621,0.018323,0.059569,0.050025,0.060484,...,0.001295,0.078571,0.0244,0.064479,0.019022,0.00166,0.01222,86706.281126,0.086706,177614.301916
2025-11-28,0.309933,0.088573,0.118116,0.075564,0.024257,0.030168,0.020852,0.097649,0.066505,0.083439,...,0.002662,0.080191,0.023416,0.062846,0.020552,0.001501,0.012304,87530.353913,0.08753,206179.681726
2025-12-31,0.299867,0.066584,0.124269,0.066584,-0.008349,0.029024,0.019727,0.07384,0.067403,0.066631,...,0.002026,0.082104,0.022359,0.061158,0.022359,0.001771,0.012421,88331.289298,0.088331,196109.309909
2026-01-30,0.321301,0.073737,0.121083,0.015284,0.015284,0.030579,0.021221,0.081586,0.07116,0.202829,...,0.002841,0.083847,0.021518,0.059186,0.001611,0.001611,0.012511,89773.700508,0.089774,214390.75964
2026-02-20,0.351808,0.139411,0.16894,0.038091,0.022463,0.03291,0.023524,0.144901,0.089171,0.306995,...,0.004062,0.085069,0.021045,0.057965,0.00274,0.001127,0.012569,90531.964969,0.090532,241670.185651



## 5. Graphiques <a id='1'></a>

### 5a. Rendements cumulés — Portefeuille vs Benchmark

In [10]:
fig_cumul = reporter.plot_cumulative_returns()
fig_cumul.show()

### 5b. Drawdowns du portefeuille

In [11]:
fig_dd = reporter.plot_drawdowns()
fig_dd.show()

### 5c. Volatilité glissante annualisée

In [12]:
fig_vol = reporter.plot_historical_volatility()
fig_vol.show()

### 5d. Corrélation glissante avec le Benchmark

In [13]:
fig_corr = reporter.plot_historical_correlation()
fig_corr.show()

### 5e. Composition du portefeuille

In [14]:

comp_date = pd.Timestamp(AS_OF_DATE)


comp_df = reporter.get_portfolio_composition(comp_date)
print(f"Portefeuille au {comp_date.date()} : {len(comp_df)} positions")
if comp_df.empty or 'Weight' not in comp_df.columns:
    print("  Composition indisponible pour cette date (probablement hors rebalancement).")
else:
    print(f"  Long  : {(comp_df['Weight'] > 0).sum()} positions")
    print(f"  Short : {(comp_df['Weight'] < 0).sum()} positions")
    print()
    display(comp_df.head())


Portefeuille au 2026-01-30 : 470 positions
  Long  : 130 positions
  Short : 145 positions



,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
0,ALLN SE Equity,ALLREAL HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,226.0,0.069481
1,MOWI NO Equity,MOWI ASA,NORWAY,NOK,Consommation de base,Produits alimentaires,221.4,0.048979
2,AKRBP NO Equity,AKER BP ASA,NORWAY,NOK,Energie,"Pétrole, gaz et combustibles",281.4,0.047872
3,MOBN SE Equity,MOBIMO HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,389.0,0.045998
4,SCMN SE Equity,SWISSCOM AG-REG,SWITZERLAND,CHF,Services de communication,Services de télécommunicatio,633.5,0.044298


In [15]:
comp_charts = reporter.plot_composition_barcharts(comp_date)

comp_charts["Sector"].show()

In [16]:
comp_charts["Country"].show()

In [17]:
comp_charts["Industry"].show()

In [18]:
comp_charts["Currency"].show()

## 6. Analyse du portefeuille

### 6a. Top 10 positions 

In [19]:
top10_weights = reporter.get_top_10_weights(comp_date)

print(f"Top 10 positions au {comp_date.date()} :")
display(top10_weights)

Top 10 positions au 2026-01-30 :


,Ticker,Name,Sector,Country,Weight
0,ALLN SE Equity,ALLREAL HOLDING AG-REG,Immobilier,SWITZERLAND,0.069481
1,ADM LN Equity,ADMIRAL GROUP PLC,Finance,BRITAIN,-0.057471
2,ULVR LN Equity,UNILEVER PLC,Consommation de base,BRITAIN,-0.055330
3,SDF GY Equity,K+S AG-REG,Matériaux,GERMANY,-0.055054
4,NTGY SQ Equity,NATURGY ENERGY GROUP SA,Services aux collectivités,SPAIN,-0.049732
5,MOWI NO Equity,MOWI ASA,Consommation de base,NORWAY,0.048979
6,FDJU FP Equity,FDJ UNITED,Consommation discrétionnaire,FRANCE,-0.048684
7,RED SQ Equity,REDEIA CORP SA,Services aux collectivités,SPAIN,-0.048048
8,AKRBP NO Equity,AKER BP ASA,Energie,NORWAY,0.047872
9,VIS SQ Equity,VISCOFAN SA,Consommation de base,SPAIN,-0.046352


### 6b. Top 10 contributions à la performance

In [20]:
top5_pos, top5_neg = reporter.get_top_10_return_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS (période {comp_date.date()}) :")
display(top5_pos)

print(f"Top 5 contributeurs NÉGATIFS (période {comp_date.date()}) :")
display(top5_neg)

Top 5 contributeurs POSITIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,SBMO NA Equity,0.012934,SBM OFFSHORE NV,Energie
1,ADM LN Equity,0.007600,ADMIRAL GROUP PLC,Finance
2,ORA FP Equity,0.005615,ORANGE,Services de communication
3,LIGHT NA Equity,0.005011,SIGNIFY NV,Industrie
4,PSON LN Equity,0.004992,PEARSON PLC,Consommation discrétionnaire


Top 5 contributeurs NÉGATIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,LOTB BB Equity,-0.011412,LOTUS BAKERIES,Consommation de base
1,BN FP Equity,-0.005882,DANONE,Consommation de base
2,VK FP Equity,-0.005726,VALLOUREC SA,Energie
3,CDI FP Equity,-0.004848,CHRISTIAN DIOR SE,Consommation discrétionnaire
4,ASM NA Equity,-0.004692,ASM INTERNATIONAL NV,Technologies de l'information


### 6c. Top 10 contributions au risque du portefeuille (MCTR)

In [21]:
top5_risk_pos, top5_risk_neg = reporter.get_top_10_risk_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS au risque au {comp_date.date()} :")
display(top5_risk_pos)

print(f"Top 5 contributeurs NÉGATIFS au risque au {comp_date.date()} :")
display(top5_risk_neg)

Top 5 contributeurs POSITIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,SDF GY Equity,0.004286,-0.055054,K+S AG-REG,Matériaux
1,BARN SE Equity,0.003644,0.029114,BARRY CALLEBAUT AG-REG,Consommation de base
2,FDJU FP Equity,0.003003,-0.048684,FDJ UNITED,Consommation discrétionnaire
3,QIA GY Equity,0.002426,-0.035366,QIAGEN N.V.,Soins de santé
4,TSCO LN Equity,0.002262,0.033178,TESCO PLC,Consommation de base


Top 5 contributeurs NÉGATIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,INCH LN Equity,-0.000409,0.016757,INCHCAPE PLC,Consommation discrétionnaire
1,CDI FP Equity,-0.000331,0.011927,CHRISTIAN DIOR SE,Consommation discrétionnaire
2,MC FP Equity,-0.000330,0.012769,LVMH MOET HENNESSY LOUIS VUI,Consommation discrétionnaire
3,BA/ LN Equity,-0.000229,0.014304,BAE SYSTEMS PLC,Industrie
4,SPIE FP Equity,-0.000159,0.012113,SPIE SA,Industrie


### 6d. Rendements cumulés calendaires du portefeuille

In [22]:
fig_calendar_returns = reporter.plot_calendar_returns_heatmap()

fig_calendar_returns.show()


## 7. Portfolio PnL

PnL cumulé du portefeuille

In [23]:
fig_pnl = reporter.plot_pnl()
fig_pnl.show()

## Exportation des Résultats

In [24]:
out_path = reporter.run_full_report(output_dir=None)
print("Reporting folder:", out_path)

out_dir = Path(out_path)

nav_files = sorted(out_dir.glob("nav_MomentumDual_*.parquet"))

metrics_files = sorted(out_dir.glob("metriques_MomentumDual_*.parquet"))

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))

bbu_files = sorted(out_dir.glob("bbu_MomentumDual_*.csv"))


[ReportingEngine] Génération du reporting dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_MinVariance_20161230_20260220
[Report] Métriques exportées : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_MinVariance_20161230_20260220/metriques_MomentumDual_MinVariance.parquet
[Report] BBU CSV exporté : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_MinVariance_20161230_20260220/bbu_MomentumDual_MinVariance.csv
[Report] Composition détaillée exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_MinVariance_20161230_20260220/composition_detaillee_MomentumDual_MinVariance.parquet
[Report] NAV exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_MinVariance_20161230_20260220/nav_MomentumDual_MinVariance.parquet
[ReportingEngine] Reporting complet généré dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_MinVariance_20161230_20260

In [25]:

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))
comp_df = pd.read_parquet(comp_files[0])
comp_df = pd.DataFrame(comp_df) 
comp_df["Date"] = pd.to_datetime(comp_df["Date"]) 
comp_files_df = comp_df[comp_df["Date"] == pd.Timestamp(AS_OF_DATE)] 
display(comp_files_df)

,Date,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
47280,2026-01-30,ALLN SE Equity,ALLREAL HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,226.00,0.069481
47281,2026-01-30,MOWI NO Equity,MOWI ASA,NORWAY,NOK,Consommation de base,Produits alimentaires,221.40,0.048979
47282,2026-01-30,AKRBP NO Equity,AKER BP ASA,NORWAY,NOK,Energie,"Pétrole, gaz et combustibles",281.40,0.047872
47283,2026-01-30,MOBN SE Equity,MOBIMO HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,389.00,0.045998
47284,2026-01-30,SCMN SE Equity,SWISSCOM AG-REG,SWITZERLAND,CHF,Services de communication,Services de télécommunicatio,633.50,0.044298
...,...,...,...,...,...,...,...,...,...
47745,2026-01-30,FDJU FP Equity,FDJ UNITED,FRANCE,EUR,Consommation discrétionnaire,"Hôtels, restaurants et loisir",22.34,-0.048684
47746,2026-01-30,NTGY SQ Equity,NATURGY ENERGY GROUP SA,SPAIN,EUR,Services aux collectivités,Gaz,26.48,-0.049732
47747,2026-01-30,SDF GY Equity,K+S AG-REG,GERMANY,EUR,Matériaux,Produits chimiques,13.82,-0.055054
47748,2026-01-30,ULVR LN Equity,UNILEVER PLC,BRITAIN,GBp,Consommation de base,Produits de soins personnels,4940.50,-0.055330


In [26]:
bbu_df = pd.read_csv(bbu_files[0])
display(bbu_df.head())

,Date,Ticker,Weights
0,2016-12-30,KESKOB FH Equity,0.044791
1,2016-12-30,MRW LN Equity,0.000000
2,2016-12-30,TSCO LN Equity,0.000000
3,2016-12-30,MOWI NO Equity,0.001769
4,2016-12-30,JMT PL Equity,0.000000
